Importing the required and necessary libraries.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [4]:
from pathlib import Path

In [5]:
path = Path("/content/cleaned_bank_churners.csv")

In [8]:
df = pd.read_csv(path)

In [10]:
df_model = df.copy()

In [11]:
df_model.head(10)

,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,0,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,0,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,0,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,0,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,0,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000
5,0,44,M,2,Graduate,Married,$40K - $60K,Blue,36,3,1,2,4010.0,1247,2763.0,1.376,1088,24,0.846,0.311
6,0,51,M,4,Unknown,Married,$120K +,Gold,46,6,1,3,34516.0,2264,32252.0,1.975,1330,31,0.722,0.066
7,0,32,M,0,High School,Unknown,$60K - $80K,Silver,27,2,2,2,29081.0,1396,27685.0,2.204,1538,36,0.714,0.048
8,0,37,M,3,Uneducated,Single,$60K - $80K,Blue,36,5,2,0,22352.0,2517,19835.0,3.355,1350,24,1.182,0.113
9,0,48,M,2,Graduate,Single,$80K - $120K,Blue,36,6,3,3,11656.0,1677,9979.0,1.524,1441,32,0.882,0.144


Inspecting "Unknown" categories:

Some categorical variables contain the value "Unknown". These values are retained because "Unknown" represents unavailable information. Removing or replacing these observations would unnecessarily remove some customers or introduce assumptions.


In [12]:
categorical_columns = ["Gender","Education_Level","Marital_Status",
    "Income_Category",
    "Card_Category"]

for column in categorical_columns:
    print(column, (df_model[column] == "Unknown").sum())

Gender 0
Education_Level 1519
Marital_Status 749
Income_Category 1112
Card_Category 0


Separating the predictors and the target:

Attrition_Flag is the target variable that indicates whether a customer is an existing or attrited customer. It is separated from the predictor variables so that there is no form of data leakage.

In [13]:
X = df_model.drop(columns=["Attrition_Flag"]).copy()
y = df_model["Attrition_Flag"].copy()

In [15]:
X.head()

,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000


In [16]:
y.head()

,Attrition_Flag
0,0
1,0
2,0
3,0
4,0


Feature 1: Average Transaction Amount

This feature explains the typical transaction size of a customer, which provides information about spending behavior that is not captured by total transaction amount alone.

In [17]:
df_model["Avg_Transaction_Amount"] = (df_model["Total_Trans_Amt"] / df_model["Total_Trans_Ct"])

In [18]:
df_model["Avg_Transaction_Amount"].head()

,Avg_Transaction_Amount
0,27.238095
1,39.121212
2,94.350000
3,58.550000
4,29.142857


Feature 2: Revolving Balance Ratio

This feature measures the proportion of available credit represented by the customer's revolving balance. A higher ratio may indicate greater reliance on revolving credit and therefore provides an additional behavioral measure that may help distinguish customer profiles where we might discover that certain types of customers: use their card very frequently, become inactive for several months, have fewer transactions, use fewer products And some of these behaviours may be associated with customers who eventually leave.

Before going into the division, we need to see if there is any credit limit that is equal to 0 as this would affect the division of the total revolving balance by the credit limit.

In [21]:
(df_model["Credit_Limit"] == 0).sum()

np.int64(0)

In [22]:
df_model["Revolving_Balance_Ratio"] = (df_model["Total_Revolving_Bal"] / df_model["Credit_Limit"])

In [23]:
df_model["Revolving_Balance_Ratio"].head()

,Revolving_Balance_Ratio
0,0.061224
1,0.104651
2,0.000000
3,0.759734
4,0.000000


Feature 3: Transaction Activity per Relationship

This provides a measure of transaction engagement relative to the customer's relationship breadth with the bank. It may help distinguish customers who are actively using their banking relationships from customers who hold several products but show relatively little transaction activity.

In [25]:
df_model["Transaction_Activity_per_Relationship"] =(
    df_model["Total_Trans_Ct"] / df_model["Total_Relationship_Count"])

In [26]:
df_model["Transaction_Activity_per_Relationship"].head()

,Transaction_Activity_per_Relationship
0,8.400000
1,5.500000
2,5.000000
3,6.666667
4,5.600000


Checking the full model to see if the newly created features are good to work with.

In [27]:
df_model.head()

,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,...,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Avg_Transaction_Amount,Revolving_Balance_Ratio,Transaction_Activity_per_Relationship
0,0,45,M,3,High School,Married,$60K - $80K,Blue,39,5,...,777,11914.0,1.335,1144,42,1.625,0.061,27.238095,0.061224,8.400000
1,0,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,...,864,7392.0,1.541,1291,33,3.714,0.105,39.121212,0.104651,5.500000
2,0,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,...,0,3418.0,2.594,1887,20,2.333,0.000,94.350000,0.000000,5.000000
3,0,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,...,2517,796.0,1.405,1171,20,2.333,0.760,58.550000,0.759734,6.666667
4,0,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,...,0,4716.0,2.175,816,28,2.500,0.000,29.142857,0.000000,5.600000


In [32]:
df_model.shape

(10127, 23)

In [28]:
print(categorical_columns)

['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']


Categorical predictors are encoded using one-hot encoding because the categories do not represent meaningful numerical distances. For example, the categories in education, income, marital status, and card type should not be treated as values such as 1, 2, 3 and 4 because doing so would incorrectly imply an ordered numerical relationship.

drop_first=True is used to remove one category from each categorical variable. The omitted category becomes the reference category, while the remaining dummy variables represent differences relative to that reference. This also helps to avoid multicollinearity for models that are sensitive to it.

In [29]:
X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True,
    dtype=int)

In [30]:
X_encoded.head()

,Customer_Age,Dependent_count,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,...,Marital_Status_Single,Marital_Status_Unknown,Income_Category_$40K - $60K,Income_Category_$60K - $80K,Income_Category_$80K - $120K,Income_Category_Less than $40K,Income_Category_Unknown,Card_Category_Gold,Card_Category_Platinum,Card_Category_Silver
0,45,3,39,5,1,3,12691.0,777,11914.0,1.335,...,0,0,0,1,0,0,0,0,0,0
1,49,5,44,6,1,2,8256.0,864,7392.0,1.541,...,1,0,0,0,0,1,0,0,0,0
2,51,3,36,4,1,0,3418.0,0,3418.0,2.594,...,0,0,0,0,1,0,0,0,0,0
3,40,4,34,3,4,1,3313.0,2517,796.0,1.405,...,0,1,0,0,0,1,0,0,0,0
4,40,3,21,5,1,0,4716.0,0,4716.0,2.175,...,0,0,0,1,0,0,0,0,0,0


In [31]:
X_encoded.shape

(10127, 32)

Confirming if the categorical_columns that are encoded.

In [33]:
X_encoded.select_dtypes(include="object").columns

Index([], dtype='object')